In [19]:
import numpy as np
import matplotlib.pyplot as plt
import math as m

from ipywidgets import interact, FloatSlider, IntSlider, Play, jslink, VBox

In [20]:
#Cambell
P = 100*60*60*24*365.25
T = 0.4*P
e = 0.5
a = 149597870700 
i = 60 #graus
w = 250 #graus
o = 120 #graus

#Problema de dos cossos
G = 6.67 * 1e-11
m1 = 2*np.pi**2*a**3/(G*P**2)


In [21]:
def trobarE(P,t,T,e):
    mu = 2*np.pi/P
    Eo = 0
    E = 1
    ε = 1e-6

    M = mu*(t-T)

    def Ef (M,Ei,e):
        return (Ei - e*m.sin(Ei) - M)/(1 - e*m.cos(Ei))
    
    while abs(E-Eo) > ε:
        Eo = E
        E = Eo - Ef(M,Eo,e)
        


    return E
        

In [22]:
def orbites (P,T,e,a,i,w,o,m1):
    t = 0
    dt = P*1e-3
    mu = 2*np.pi/P
    E = 0
    ε = 1e-10
    G = 6.67 * 1e-11
    m2 = 4*np.pi**2*a**3/(G*P**2) - m1
    x1_list = []
    y1_list = []
    x2_list = []
    y2_list = []
    t_list=[]

    
    c1 = m2/(m1 + m2)
    c2 = - m1/(m1 + m2)

    A = a*(np.cos(m.radians(o))*np.cos(np.radians(w)) - np.sin(np.radians(o))*np.sin(np.radians(w))*np.cos(np.radians(i)))
    B = a*(np.sin(m.radians(o))*np.cos(np.radians(w)) + np.cos(np.radians(o))*np.sin(np.radians(w))*np.cos(np.radians(i)))
    F = a*(-np.cos(m.radians(o))*np.sin(np.radians(w)) - np.sin(np.radians(o))*np.cos(np.radians(w))*np.cos(np.radians(i)))
    G = a*(-np.sin(m.radians(o))*np.sin(np.radians(w)) + np.cos(np.radians(o))*np.cos(np.radians(w))*np.cos(np.radians(i)))
                
    while t < P:
        E = trobarE(P,t,T,e)

        X = np.cos(E) - e
        Y = np.sqrt(1-e**2) * np.sin(E)


        x = A*X + F*Y
        y = B*X + G*Y

        x1 = c1*x
        x2 = c2*x
        y1 = c1*y
        y2 = c2*y        


        x1_list.append(x1)
        y1_list.append(y1)
        x2_list.append(x2)
        y2_list.append(y2)
        t_list.append(t)

        t = t+dt

    return x1_list, y1_list, x2_list, y2_list, t_list

In [23]:
def plot_orbites(P,T,e,a,i,w,o,m1):
    x1_list, y1_list, x2_list, y2_list, t_list = orbites(P,T,e,a,i,w,o,m1)

    x1 = x1_list[-1]
    y1 = y1_list[-1]
    x2 = x2_list[-1]
    y2 = y2_list[-1]


    plt.scatter (x1_list, y1_list, c=t_list, cmap = 'pink', s = 5, zorder = 1)
    plt.plot (x1, y1, '*', color = 'yellow', markersize = 15, zorder = 3)
    plt.scatter (x2_list, y2_list, c=t_list, cmap = 'bone', s = 5, zorder = 1)
    plt.plot (x2, y2, '*', color = 'yellow', markersize = 15, zorder = 3)
    plt.gca().set_aspect('equal')
    plt.show()


In [24]:
interact(
    plot_orbites,
    P = P,
    T = FloatSlider(min=0, max=P, value=T),
    e = FloatSlider(min=0, max=0.999999999, step=0.1, value=e),
    a = a,
    i = FloatSlider(min=0, max=360, value=i),
    w = FloatSlider(min=0, max=360, value=w),
    o = FloatSlider(min=0, max=360, value=o),
    m1 = FloatSlider(min=0, max=4*np.pi**2*a**3/(G*P**2), value=m1)
)


interactive(children=(FloatSlider(value=3155760000.0, description='P', max=9467280000.0, min=-3155760000.0), F…

<function __main__.plot_orbites(P, T, e, a, i, w, o, m1)>

In [25]:
x1_list, y1_list, x2_list, y2_list, t_list = orbites(P,T,e,a,i,w,o,m1)

def plot_frame(frame):
    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    # punts actuals
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15)
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15)

    plt.gca().set_aspect('equal')
    plt.show()

# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=len(t_list)-1,
    step=1,
    interval=30
)

slider = IntSlider(
    min=0,
    max=len(t_list)-1,
    value=0,
    step=1
)

jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot_frame, frame=slider)

display(ui, out)


interactive(children=(IntSlider(value=0, description='frame', max=999), Output()), _dom_classes=('widget-inter…

<function __main__.plot_frame(frame)>

In [26]:
def plot_frame_mod(frame,P,T,e,a,i,w,o,m1):
    x1_list, y1_list, x2_list, y2_list, t_list = orbites(P,T,e,a,i,w,o,m1)

    plt.figure(figsize=(6,6))

    plt.scatter(x1_list[:frame], y1_list[:frame], c=t_list[:frame], cmap = 'pink', s=5)
    plt.scatter(x2_list[:frame], y2_list[:frame], c=t_list[:frame], cmap = 'bone', s=5)

    # punts actuals
    plt.plot(x1_list[frame], y1_list[frame], '*', color='yellow', markersize=15)
    plt.plot(x2_list[frame], y2_list[frame], '*', color='yellow', markersize=15)

    plt.gca().set_aspect('equal')
    plt.show()

# widgets de reproducció
play = Play(
    value=0,
    min=0,
    max=len(t_list)-1,
    step=1,
    interval=30
)



jslink((play, 'value'), (slider, 'value'))

ui = VBox([play, slider])

out = interact(plot_frame_mod, frame=slider,
    P = FloatSlider(min=T, max = 3*P, value=P),
    T = FloatSlider(min=0, max=P, value=T),
    e = FloatSlider(min=0, max=0.999999999, step=0.1, value=e),
    a = a,
    i = FloatSlider(min=0, max=360, value=i),
    w = FloatSlider(min=0, max=360, value=w),
    o = FloatSlider(min=0, max=360, value=o),
    m1 = FloatSlider(min=0, max=4*np.pi**2*a**3/(G*P**2), value=m1))

display(ui, out)

interactive(children=(IntSlider(value=0, description='frame', max=999), FloatSlider(value=3155760000.0, descri…

<function __main__.plot_frame_mod(frame, P, T, e, a, i, w, o, m1)>